# Person 3 — Argument Pattern-Matcher (Layer 1 + Layer 2)

Builds and validates the argument pattern-matcher described in `PERSON3_context.md`,
using the rule files from Person 1 (Art 21/32/14) and Person 2 (Art 19/20/22).

**Before running this notebook:**
1. Restore your `.backup` export into the new Aura instance — on the instance's page in
   `console.neo4j.io`, use the **Import Database** tab, drag in the `.backup` file, and
   click Upload (works directly for files under 4GB, no code needed). Wait for the
   instance to leave "Loading" state.
2. Grab the new instance's **Connection URI** and the password you set for it — you'll
   paste both into the cell below.
3. Have `rules_art21_32_14.json` and `rules_art19_20_22_v2.json` ready to upload when
   prompted (Cell 3).

Run cells top to bottom. Cells that only matter *after* you've had a teammate label the
validation sheet are marked and can be skipped until then.


## Setup

In [5]:
!pip install -q neo4j sentence-transformers pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 6.9 MB/s eta 0:00:00


In [6]:
from neo4j import GraphDatabase

URI = "neo4j+s://dfed4893.databases.neo4j.io"  # from the new instance's Connection URI
USER = "neo4j"
#33cf37b9
#wHYcuO-2HaGKL5ZLTrD1orN-6Df7OKK47FaH_uHTCHc
#PASSWORD = "AyGQQOdAig7Qhjw9Z_lpmonX-ZGiFviF1q8MKUMEwiQ"
PASSWORD = "0iOSmjsbDmvX5rF_1b9ksvP0mn-fQ8Qm2F5aVCDaqt0"                     # set when you created the instance

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("Connected.")

Connected.


## Step 1 — Upload and merge the two rule files

Upload `rules_art21_32_14.json` (Person 1 / Shrikri) and `rules_art19_20_22_v2.json`
(Person 2 / Bhumika) when the file picker appears.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select both rule JSONs

Saving rules_art21_32_14.json to rules_art21_32_14.json
Saving rules_art19_20_22_v2.json to rules_art19_20_22_v2.json


In [ ]:
import json, re

def load_and_merge(p1_path, p2_path):
    with open(p1_path) as f:
        p1 = json.load(f)
    with open(p2_path) as f:
        p2 = json.load(f)

    rules = []
    for r in p1["rules"]:
        r = dict(r)
        r["source"] = "person1_shrikri"
        r.setdefault("constitutional_text_paraphrase", None)  # missing in P1 — flagged for backfill
        rules.append(r)

    for key in ("article_19_rules", "article_20_rules", "article_22_rules"):
        for r in p2.get(key, []):
            r = dict(r)
            r["source"] = "person2_bhumika"
            rules.append(r)

    ids = [r["rule_id"] for r in rules]
    dupes = {i for i in ids if ids.count(i) > 1}
    if dupes:
        raise ValueError(f"Duplicate rule_id(s): {dupes}")

    # normalize condition_logic.requires into a structured, machine-usable form
    for r in rules:
        all_of, any_of, req_concepts, manual_notes = [], [], [], []
        for req in r["condition_logic"].get("requires", []):
            req = req.strip()
            if req.lower().startswith("co-occurs-with"):
                manual_notes.append(req); continue
            if req.upper().startswith("APPLIES_CONCEPT"):
                req_concepts.append(req.split(" ", 1)[1]); continue
            stids = re.findall(r"Art_[A-Za-z0-9_]+", req)
            if " OR " in req:
                any_of.extend(f"Constitution_{s}" for s in stids)
            elif stids:
                all_of.extend(f"Constitution_{s}" for s in stids)
            else:
                manual_notes.append(req)
        r["_structured_requires"] = {
            "all_of_statutes": all_of, "any_of_statutes": any_of,
            "requires_concepts": req_concepts, "manual_notes": manual_notes,
        }
    return rules

RULES = load_and_merge("rules_art21_32_14.json", "rules_art19_20_22_v2.json")
print(f"Merged {len(RULES)} rules.")
with open("merged_rules_v1.json", "w") as f:
    json.dump(RULES, f, indent=2)

Merged 31 rules.


## Step 2 — Sanity-check the merged rules against the live graph

Confirms whether Article 19/20/22 statute nodes actually exist at sub-clause
granularity, and which `applicable_concepts` have no matching `LegalConcept` node
yet (expected for many — see the feedback notes from earlier).

In [ ]:
with driver.session() as session:
    print("=== Statute granularity for Art 19/20/22 ===")
    for row in session.run("""
        MATCH (s:Statute)
        WHERE s.statute_id CONTAINS 'Art_19' OR s.statute_id CONTAINS 'Art_20' OR s.statute_id CONTAINS 'Art_22'
        RETURN DISTINCT s.statute_id AS sid ORDER BY sid
    """):
        print(" ", row["sid"])

    print("\n=== Current LegalConcept nodes ===")
    existing_concepts = {r["name"] for r in session.run("MATCH (lc:LegalConcept) RETURN lc.name AS name")}

missing = {}
for r in RULES:
    for c in r.get("applicable_concepts", []):
        if c not in existing_concepts:
            missing.setdefault(c, []).append(r["rule_id"])
print(f"\n{len(missing)} applicable_concepts have no matching LegalConcept node right now.")
for c, rids in sorted(missing.items())[:10]:
    print(f"  '{c}' <- {rids}")
if len(missing) > 10:
    print(f"  ... and {len(missing) - 10} more")

=== Statute granularity for Art 19/20/22 ===
  Constitution_Art_19
  Constitution_Art_19(0)
  Constitution_Art_19(1)
  Constitution_Art_19(1)(1)
  Constitution_Art_19(1)(2)
  Constitution_Art_19(1)(3)
  Constitution_Art_19(1)(9)
  Constitution_Art_19(1)(a)
  Constitution_Art_19(1)(b)
  Constitution_Art_19(1)(b)(c)
  Constitution_Art_19(1)(b)(c)(d)
  Constitution_Art_19(1)(c)
  Constitution_Art_19(1)(c)(g)
  Constitution_Art_19(1)(d)
  Constitution_Art_19(1)(e)
  Constitution_Art_19(1)(f)
  Constitution_Art_19(1)(g)
  Constitution_Art_19(1)(i)
  Constitution_Art_19(1)(j)
  Constitution_Art_19(2)
  Constitution_Art_19(2)(3)
  Constitution_Art_19(2)(3)(4)(5)
  Constitution_Art_19(3)
  Constitution_Art_19(4)
  Constitution_Art_19(5)
  Constitution_Art_19(6)
  Constitution_Art_19(6)(i)
  Constitution_Art_19(a)
  Constitution_Art_19(b)
  Constitution_Art_19(d)
  Constitution_Art_19(e)
  Constitution_Art_19(f)
  Constitution_Art_19(g)
  Constitution_Art_19(i)
  Constitution_Art_19(i)(f)
  Con

## Step 3 — Layer 1 (keyword) + Layer 2 (semantic) matcher

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # same model as compute_similar_facts.py

def keyword_hit(text: str, rule: dict) -> bool:
    low = (text or "").lower()
    return any(kw.lower() in low for kw in rule.get("layer1_keywords", []))

exemplar_texts, exemplar_rule_idx = [], []
for i, r in enumerate(RULES):
    for ex in r["exemplar_sentences"]:
        exemplar_texts.append(ex)
        exemplar_rule_idx.append(i)
exemplar_rule_idx = np.array(exemplar_rule_idx)

exemplar_emb = MODEL.encode(exemplar_texts, normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=True)
print(f"Embedded {len(exemplar_texts)} exemplar sentences across {len(RULES)} rules.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedded 159 exemplar sentences across 31 rules.


In [ ]:
def score_arguments(arg_texts, batch_size=2000):
    """Top-2-average cosine similarity of each argument text against each rule's exemplars."""
    n_rules, n_args = len(RULES), len(arg_texts)
    best_score = np.zeros((n_args, n_rules), dtype=np.float32)
    best_exemplar_idx = np.full((n_args, n_rules), -1, dtype=np.int32)
    for start in range(0, n_args, batch_size):
        batch = arg_texts[start:start + batch_size]
        arg_emb = MODEL.encode(batch, normalize_embeddings=True, convert_to_numpy=True)
        sims = arg_emb @ exemplar_emb.T
        for ridx in range(n_rules):
            cols = np.where(exemplar_rule_idx == ridx)[0]
            if len(cols) == 0:
                continue
            rule_sims = sims[:, cols]
            top2 = np.sort(rule_sims, axis=1)[:, -2:] if rule_sims.shape[1] >= 2 else rule_sims
            best_score[start:start + len(batch), ridx] = top2.mean(axis=1)
            best_exemplar_idx[start:start + len(batch), ridx] = cols[rule_sims.argmax(axis=1)]
    return best_score, best_exemplar_idx

## Step 4 — Run the matcher over petitioner arguments in the graph

Produces one record per `(case_id, rule_id)`, per the output format in
`PERSON3_context.md`.

In [ ]:
with driver.session() as session:
    rows = [dict(r) for r in session.run("""
        MATCH (c:Case)-[:MAKES_ARGUMENT]->(a:Argument)
        WHERE a.side = 'petitioner' AND a.text IS NOT NULL AND a.text <> ''
        RETURN c.case_id AS case_id, a.argument_id AS argument_id, a.text AS text
    """)]
print(f"Pulled {len(rows)} petitioner argument sentences.")

texts = [r["text"] for r in rows]
scores, exemplar_idx = score_arguments(texts)

best = {}
for i, row in enumerate(rows):
    for ridx, rule in enumerate(RULES):
        s = float(scores[i, ridx])
        key = (row["case_id"], rule["rule_id"])
        if key not in best or s > best[key]["similarity_score"]:
            ei = exemplar_idx[i, ridx]
            best[key] = {
                "case_id": row["case_id"], "rule_id": rule["rule_id"],
                "keyword_hit": keyword_hit(row["text"], rule),
                "similarity_score": round(s, 4),
                "matched_exemplar": exemplar_texts[ei] if ei >= 0 else None,
                "matched_argument_id": row["argument_id"],
            }

with open("argument_pattern_matches.json", "w") as f:
    json.dump(list(best.values()), f, indent=2)
print(f"Wrote {len(best)} (case_id, rule_id) records to argument_pattern_matches.json")

files.download("argument_pattern_matches.json")

Pulled 44089 petitioner argument sentences.
Wrote 747751 (case_id, rule_id) records to argument_pattern_matches.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from google.colab import files
uploaded = files.upload()  # select matcher_labeling_sheet_labeled.xlsx

import json, pandas as pd

with open("argument_pattern_matches.json") as f:
    data = json.load(f)

# don't re-ask you to label what you already labeled
old_labeled = pd.read_excel("matcher_labeling_sheet_labeled.xlsx")  # re-upload if needed
already_labeled = set(zip(old_labeled["case_id"], old_labeled["rule_id"]))

by_rule = {}
for m in data:
    by_rule.setdefault(m["rule_id"], []).append(m)

top_k = []
for rid, ms in by_rule.items():
    picked = 0
    for m in sorted(ms, key=lambda m: -m["similarity_score"]):
        if (m["case_id"], m["rule_id"]) in already_labeled:
            continue
        top_k.append(m)
        picked += 1
        if picked >= 5:
            break

print(f"{len(top_k)} new high-confidence candidates to label")

ids = [m["matched_argument_id"] for m in top_k]
with driver.session() as session:
    text_map = {r["aid"]: r["text"] for r in session.run(
        "MATCH (a:Argument) WHERE a.argument_id IN $ids RETURN a.argument_id AS aid, a.text AS text", ids=ids)}

topk_df = pd.DataFrame(top_k)
topk_df["argument_text"] = topk_df["matched_argument_id"].map(text_map)
topk_df["label"] = ""
topk_df = topk_df[["case_id","rule_id","argument_text","matched_exemplar","similarity_score","keyword_hit","matched_argument_id","label"]]
topk_df.to_excel("matcher_labeling_sheet_topk.xlsx", index=False)

from google.colab import files
files.download("matcher_labeling_sheet_topk.xlsx")

155 new high-confidence candidates to label


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 5 — Build the validation labeling sheet

Sample ~100 matches spread across rules, pull the argument text back in, and export
a spreadsheet. Hand this to a teammate who *didn't* build the matcher — have them fill
the `label` column with `yes` / `no` / `ambiguous`, blind to the similarity score.

In [ ]:
import random, pandas as pd
random.seed(42)

by_rule = {}
for m in best.values():
    by_rule.setdefault(m["rule_id"], []).append(m)

per_rule_n = max(1, 90 // len(by_rule))
sample = [m for ms in by_rule.values() for m in random.sample(ms, min(per_rule_n, len(ms)))][:100]

ids = [m["matched_argument_id"] for m in sample]
with driver.session() as session:
    text_map = {r["aid"]: r["text"] for r in session.run(
        "MATCH (a:Argument) WHERE a.argument_id IN $ids RETURN a.argument_id AS aid, a.text AS text", ids=ids)}

df = pd.DataFrame(sample)
df["argument_text"] = df["matched_argument_id"].map(text_map)
df["label"] = ""  # teammate fills: yes / no / ambiguous
df = df[["case_id", "rule_id", "argument_text", "matched_exemplar", "similarity_score", "keyword_hit", "matched_argument_id", "label"]]
df.to_excel("matcher_labeling_sheet.xlsx", index=False)
print(f"Wrote matcher_labeling_sheet.xlsx with {len(df)} rows to label.")

files.download("matcher_labeling_sheet.xlsx")

Wrote matcher_labeling_sheet.xlsx with 62 rows to label.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 6 — Score against labels (run this later, once the sheet comes back)

Skip this section until a teammate has filled in the `label` column and sent the file
back. Upload the labeled version when prompted, then run the threshold sweep.

In [ ]:
labeled_upload = files.upload()  # upload the LABELED version of matcher_labeling_sheet.xlsx
labeled_filename = list(labeled_upload.keys())[0]

labeled = pd.read_excel(labeled_filename)
labeled = labeled[labeled["label"].isin(["yes", "no"])]  # drop ambiguous from threshold tuning
labeled["y_true"] = (labeled["label"] == "yes").astype(int)

rows_ = []
for t in np.arange(0.30, 0.85, 0.05):
    pred = (labeled["similarity_score"] >= t).astype(int)
    tp = ((labeled.y_true == 1) & (pred == 1)).sum()
    fp = ((labeled.y_true == 0) & (pred == 1)).sum()
    fn = ((labeled.y_true == 1) & (pred == 0)).sum()
    p = tp / (tp + fp) if (tp + fp) else 0
    r_ = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * p * r_ / (p + r_) if (p + r_) else 0
    rows_.append({"threshold": round(t, 2), "precision": round(p, 3), "recall": round(r_, 3), "f1": round(f1, 3), "n": len(labeled)})

report = pd.DataFrame(rows_)
print(report.to_string(index=False))
print(f"\nBest by F1: {report.loc[report.f1.idxmax()].to_dict()}")
print("\nNote: this sample is spread thin across 31 rules — treat this as a single")
print("global threshold (or per-domain at most) for now, and re-validate with more")
print("labels once the v_n-based KG rebuild lands.")

KeyboardInterrupt: 